# Introduction to R Syntax for Criminology Analysis
## Solution Notebook — Prioritizing High-Impact Crime Response

Complete working solutions for every step. Outputs shown as comments / expected results (computed from the same synthetic data).

**Flowchart of the desired outcome:**

![Criminology R Pipeline](criminology_r_syntax_flowchart.png)


## 0. Setup — Load required libraries

In [ ]:
library(dplyr)
library(readr)
# Both libraries loaded successfully.


## 1. Load the crime data

In [ ]:
crimes <- read_csv("data/crimes.csv")
# Parsed with column specification ...
# Rows: 40   Columns: 9


## 2. Inspect the data frame

In [ ]:
crimes          # full tibble print
head(crimes)    # first 6 rows
summary(crimes) # numeric summaries + factor levels for type/district/solved
# Expected: monthly_incidents mean ~15 700, severity mean ~5, 8 unique types, etc.


## 3. Unique crime types

In [ ]:
unique_types <- unique(crimes$type)
print(unique_types)
# [1] "Theft" "Fraud" "Homicide" "Assault" "Vandalism" "Robbery" "Drug Offense" "Burglary"


## 4. Core dplyr pipeline — high-priority non-theft crimes

In [ ]:
high_priority <- crimes %>%
  select(-district, -year_reported, -victims) %>%
  filter(monthly_incidents > 15000, type != "Theft") %>%
  arrange(desc(media_mentions))

print(high_priority)
# Expected ~15 rows, top rows often Drug Offense / Assault / Homicide with media_mentions > 3000
# (exact order depends on random seed; first few: C1035, C1006, C1002, C1004, ...)


## 5. Math helpers — sqrt, floor, ceiling

In [ ]:
# Example on first severity
example_severity <- crimes$severity_score[1]
solution <- sqrt(example_severity)
print(solution)          # ~1.97 for severity 3.9

round_down <- floor(3.14)
round_up   <- ceiling(3.14)
print(round_down)        # 3
print(round_up)          # 4

# Practical use: estimate patrol units
units_conservative <- floor(high_priority$monthly_incidents / 5000)
units_optimistic   <- ceiling(high_priority$monthly_incidents / 5000)
print(head(units_conservative))
print(head(units_optimistic))


## 6. Comparisons and if-else decision logic

In [ ]:
# Simple comparisons (original style)
print(56 >= 129)   # FALSE
print(56 != 129)   # TRUE

# Decision for a representative case (first high-priority row)
row <- high_priority[1, ]
message <- "Standard response"
high_incidents <- row$monthly_incidents > 15000
is_unsolved    <- !row$solved

if (high_incidents && is_unsolved) {
  message <- "Dispatch priority units!"
} else {
  message <- "Standard response"
}
print(message)
# For most top rows that are unsolved → "Dispatch priority units!"


## 7. Variables and vectors

In [ ]:
name <- "Jeremias"
years_service <- 23
crime_codes <- c(501, 333, 4444)   # radio / offence codes

print(name)
print(years_service)
print(crime_codes)
# "Jeremias"
# 23
# 501 333 4444


## 8. Arithmetic — impact “volume”

In [ ]:
# Pure multiplication practice (cube volume style)
impact_volume <- 3 * 3 * 3
print(impact_volume)   # 27

# More meaningful composite for a chosen high-priority case
r <- high_priority[1, ]
composite_impact <- r$severity_score * (r$monthly_incidents / 1000) * 1.5
print(composite_impact)


## 9. Explicit print statements

In [ ]:
print("Priority analysis complete")
print(23)
print("23")   # character vs numeric distinction


## Alternate Code Paths

In [ ]:
# Base-R equivalent (no pipe, no dplyr)
cols_keep <- setdiff(names(crimes), c("district", "year_reported", "victims"))
tmp <- crimes[, cols_keep]
high_priority_base <- tmp[tmp$monthly_incidents > 15000 & tmp$type != "Theft", ]
high_priority_base <- high_priority_base[order(-high_priority_base$media_mentions), ]
print(head(high_priority_base))

# Alternative threshold / arrange by severity instead of media
alt <- crimes %>%
  select(-district, -year_reported) %>%
  filter(monthly_incidents > 12000, type %in% c("Homicide", "Robbery", "Assault")) %>%
  arrange(desc(severity_score))
print(alt)


## More Practice — Solutions

In [ ]:
# 1. Count unsolved high-priority
n_unsolved <- high_priority %>% filter(!solved) %>% nrow()
print(paste("Unsolved high-priority cases:", n_unsolved))
# Expected around 12

# 2. mutate units_needed
with_units <- high_priority %>%
  mutate(units_needed = ceiling(monthly_incidents / 4000))
print(with_units %>% select(crime_id, type, monthly_incidents, units_needed) %>% head())

# 3. Homicide / Robbery only, by severity
violent <- crimes %>%
  filter(type %in% c("Homicide", "Robbery")) %>%
  arrange(desc(severity_score))
print(violent)


## Simulation Section — Tunable Thresholds (Solution)

In [ ]:
# Parameterised simulation — change these three values and re-run
incident_threshold <- 15000
exclude_types      <- c("Theft")
available_officers <- 12

sim_priority <- crimes %>%
  filter(monthly_incidents > incident_threshold,
         !(type %in% exclude_types)) %>%
  arrange(desc(media_mentions))

n_priority <- nrow(sim_priority)
allocated  <- min(available_officers, n_priority)

print(paste("Threshold:", incident_threshold,
            "| Excluded:", paste(exclude_types, collapse=", "),
            "| Priority cases:", n_priority,
            "| Officers allocated:", allocated))

# Quick sensitivity: what if threshold rises to 20000?
n_high <- crimes %>%
  filter(monthly_incidents > 20000, !(type %in% exclude_types)) %>%
  nrow()
print(paste("With threshold 20000 → priority cases:", n_high))


## Key Takeaways from the Solution

- The original Introduction-to-R-Syntax building blocks (unique, sqrt/floor/ceiling, if-else, vectors, arithmetic, dplyr select/filter/arrange) are sufficient to turn a raw incident table into an actionable priority list.
- Small changes to a numeric threshold or exclusion list can materially change resource demand — hence the simulation cell.
- Always keep the *audience* of the final list in mind (field commanders need the ranked IDs; executives need the count and the “why”).
